In [2]:
import argparse
import datetime
import json
import numpy as np
import os
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from torch.utils.tensorboard import SummaryWriter

import timm

assert timm.__version__ == "0.3.2" # version check
from timm.models.layers import trunc_normal_
from timm.data.mixup import Mixup
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from timm.models.layers import to_2tuple

import util.lr_decay as lrd
import util.misc as misc
from util.datasets import build_dataset
from util.pos_embed import interpolate_pos_embed, interpolate_pos_embed_audio, interpolate_patch_embed_audio, interpolate_pos_embed_img2audio
from util.misc import NativeScalerWithGradNormCount as NativeScaler

from src import models_vit

from src.engine_finetune import train_one_epoch, evaluate, val_one_epoch  #, train_one_epoch_av, evaluate_av
from src.dataset import AudiosetDataset, DistributedWeightedSampler, DistributedSamplerWrapper
from timm.models.vision_transformer import PatchEmbed

from torch.utils.data import WeightedRandomSampler

from torch.utils.data import Sampler

# New
from src.AEdataset import BirdsongDataset
from src.AEdataset import generatePCNEMelSpec

from pathlib import Path

current_working_directory = Path.cwd()
script_dir = current_working_directory.joinpath('code')

#
import librosa
import glob
from tqdm import tqdm
import torch.nn.functional as F

import csv, os, sys
import json
import torchaudio
import numpy as np
import torch.nn.functional
from torch.utils.data import Dataset, Sampler
from torch.utils.data import DistributedSampler, WeightedRandomSampler
import torch.distributed as dist
import random
import math
import psycopg2
import pandas as pd

In [2]:
model_pth_path = str(Path.cwd().parents[0].joinpath('output_finetune_model', 'model_01_1_1_1.pth'))

Audio_path = str(Path.cwd().parents[2].joinpath('Audio_data', 'finetune_Audio_mono'))

testing_list_path = str(Path.cwd().parents[2].joinpath('Audio_data', 'test_list.txt'))

output_path = str(Path.cwd().parents[0].joinpath('report', 'model_new', 'inference', 'Raw_Output'))
os.makedirs(output_path, exist_ok=True)

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, segments, audio_conf, use_fbank=False, fbank_dir=None):
        self.segments = segments
        self.use_fbank = use_fbank
        self.fbank_dir = fbank_dir
        self.audio_conf = audio_conf
        self.melbins = self.audio_conf.get('num_mel_bins')
        self.frame_shift_ms = self.audio_conf.get('frame_shift_ms')
        self.norm_mean = self.audio_conf.get('mean')
        self.norm_std = self.audio_conf.get('std')
        self.low_freq = self.audio_conf.get('low_freq')
        self.high_freq = self.audio_conf.get('high_freq')

    def _wav2fbank(self, filename, start_time=None, end_time=None):
        waveform, sr = torchaudio.load(filename)
        if start_time is not None or end_time is not None:
            start_sample = int(start_time * sr) if start_time else 0
            end_sample = int(end_time * sr) if end_time else None
            waveform = waveform[:, start_sample:end_sample]
        waveform = waveform - waveform.mean()

        fbank = torchaudio.compliance.kaldi.fbank(waveform,
                                                htk_compat=True,
                                                sample_frequency=sr,
                                                use_energy=False,
                                                window_type='hanning',
                                                num_mel_bins=self.melbins,
                                                dither=0.0,
                                                frame_shift=self.frame_shift_ms,
                                                low_freq=self.low_freq,
                                                high_freq=self.high_freq)

        target_length = self.audio_conf.get('target_length')
        n_frames = fbank.shape[0]
        p = target_length - n_frames

        use_interpolation = True
        if p != 0:
            if use_interpolation:
                fbank = fbank.unsqueeze(0).unsqueeze(0)
                fbank = F.interpolate(fbank, size=(target_length, self.melbins), mode='nearest')
                fbank = fbank.squeeze(0).squeeze(0)
            else:
                if p > 0:
                    m = torch.nn.ZeroPad2d((0, 0, 0, p))
                    fbank = m(fbank)
                elif p < 0:
                    fbank = fbank[:target_length, :]

        return fbank

    def _fbank(self, filename, start_time=None, end_time=None):
        fn1 = os.path.join(self.fbank_dir, os.path.basename(filename).replace('.wav','.npy'))
        fbank = np.load(fn1)
        if start_time is not None or end_time is not None:
            start_frame = int(start_time * 1000 / self.frame_shift_ms) if start_time else 0
            end_frame = int(end_time * 1000 / self.frame_shift_ms) if end_time else None
            fbank = fbank[start_frame:end_frame, :]

        return torch.from_numpy(fbank)

    def __getitem__(self, index):
        segment = self.segments[index]
        start_time = segment.get('start_time', 0)
        end_time = segment.get('end_time', None)

        if not self.use_fbank:
            fbank = self._wav2fbank(segment['wav'], start_time, end_time)
        else:
            fbank = self._fbank(segment['wav'], start_time, end_time)

        fbank = fbank.transpose(0,1).unsqueeze(0) # 1, 128, 1024 (...,freq,time)
        fbank = torch.transpose(fbank.squeeze(), 0, 1) # time, freq
        fbank = (fbank - self.norm_mean) / (self.norm_std * 2)

        return fbank.unsqueeze(0), segment['wav']

    def __len__(self):
        return len(self.segments)

def get_audio_duration(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    duration = waveform.shape[1] / sample_rate
    return duration

def load_input_data(audio_path, audio_conf):
    audio_duration = get_audio_duration(audio_path)
    segments = generate_segments(audio_path, audio_duration)

    dataset = InferenceDataset(segments=segments, audio_conf=audio_conf)
    return dataset

def generate_segments(audio_path, audio_duration, segment_duration=1, gap=0.25):
    num_segments = int(np.ceil((audio_duration - segment_duration) / gap))
    segments = []

    for i in range(num_segments):
        start_time = i * gap
        end_time = start_time + segment_duration
        segments.append({
            "wav": audio_path,
            "start_time": start_time,
            "end_time": end_time
        })

    return segments


class PatchEmbed_new(nn.Module):
    """ Flexible Image to Patch Embedding
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768, stride=10):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        stride = to_2tuple(stride)
        
        self.img_size = img_size
        self.patch_size = patch_size
        

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride) # with overlapped patches
        #self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

        #self.patch_hw = (img_size[1] // patch_size[1], img_size[0] // patch_size[0])
        #self.num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        _, _, h, w = self.get_output_shape(img_size) # n, emb_dim, h, w
        self.patch_hw = (h, w)
        self.num_patches = h*w

    def get_output_shape(self, img_size):
        # todo: don't be lazy..
        return self.proj(torch.randn(1,1,img_size[0],img_size[1])).shape 

    def forward(self, x):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        #assert H == self.img_size[0] and W == self.img_size[1], \
        #    f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

In [11]:
class InferenceDataset(Dataset):
    def __init__(self, segments, audio_conf, use_fbank=False, fbank_dir=None):
        self.segments = segments
        self.use_fbank = use_fbank
        self.fbank_dir = fbank_dir
        self.audio_conf = audio_conf
        self.melbins = self.audio_conf.get('num_mel_bins')
        self.frame_shift_ms = self.audio_conf.get('frame_shift_ms')
        self.norm_mean = self.audio_conf.get('mean')
        self.norm_std = self.audio_conf.get('std')
        self.low_freq = self.audio_conf.get('low_freq')
        self.high_freq = self.audio_conf.get('high_freq')

    def _wav2fbank(self, filename, start_time=None, end_time=None):
        waveform, sr = torchaudio.load(filename)
        if start_time is not None or end_time is not None:
            start_sample = int(start_time * sr) if start_time else 0
            end_sample = int(end_time * sr) if end_time else None
            waveform = waveform[:, start_sample:end_sample]
        waveform = waveform - waveform.mean()

        fbank = torchaudio.compliance.kaldi.fbank(waveform,
                                                htk_compat=True,
                                                sample_frequency=sr,
                                                use_energy=False,
                                                window_type='hanning',
                                                num_mel_bins=self.melbins,
                                                dither=0.0,
                                                frame_shift=self.frame_shift_ms,
                                                low_freq=self.low_freq,
                                                high_freq=self.high_freq)

        target_length = self.audio_conf.get('target_length')
        n_frames = fbank.shape[0]
        p = target_length - n_frames

        use_interpolation = True
        if p != 0:
            if use_interpolation:
                fbank = fbank.unsqueeze(0).unsqueeze(0)
                fbank = F.interpolate(fbank, size=(target_length, self.melbins), mode='nearest')
                fbank = fbank.squeeze(0).squeeze(0)
            else:
                if p > 0:
                    m = torch.nn.ZeroPad2d((0, 0, 0, p))
                    fbank = m(fbank)
                elif p < 0:
                    fbank = fbank[:target_length, :]

        return fbank

    def _fbank(self, filename, start_time=None, end_time=None):
        fn1 = os.path.join(self.fbank_dir, os.path.basename(filename).replace('.wav','.npy'))
        fbank = np.load(fn1)
        if start_time is not None or end_time is not None:
            start_frame = int(start_time * 1000 / self.frame_shift_ms) if start_time else 0
            end_frame = int(end_time * 1000 / self.frame_shift_ms) if end_time else None
            fbank = fbank[start_frame:end_frame, :]

        return torch.from_numpy(fbank)

    def __getitem__(self, index):
        segment = self.segments[index]
        start_time = segment.get('start_time', 0)
        end_time = segment.get('end_time', None)

        if not self.use_fbank:
            fbank = self._wav2fbank(segment['wav'], start_time, end_time)
        else:
            fbank = self._fbank(segment['wav'], start_time, end_time)

        fbank = fbank.transpose(0,1).unsqueeze(0) # 1, 128, 1024 (...,freq,time)
        fbank = torch.transpose(fbank.squeeze(), 0, 1) # time, freq
        fbank = (fbank - self.norm_mean) / (self.norm_std * 2)

        return fbank.unsqueeze(0), segment['wav']

    def __len__(self):
        return len(self.segments)

def get_audio_duration(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    duration = waveform.shape[1] / sample_rate
    return duration

def load_input_data(audio_path, audio_conf):
    audio_duration = get_audio_duration(audio_path)
    segments = generate_segments(audio_path, audio_duration)

    dataset = InferenceDataset(segments=segments, audio_conf=audio_conf)
    return dataset

def generate_segments(audio_path, audio_duration, segment_duration=1, gap=0.25):
    num_segments = int(np.ceil((audio_duration - segment_duration) / gap))
    segments = []

    for i in range(num_segments):
        start_time = i * gap
        end_time = start_time + segment_duration
        segments.append({
            "wav": audio_path,
            "start_time": start_time,
            "end_time": end_time
        })

    return segments

class PatchEmbed_new(nn.Module):
    """ Flexible Image to Patch Embedding
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768, stride=10):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        stride = to_2tuple(stride)
        
        self.img_size = img_size
        self.patch_size = patch_size
        

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride) # with overlapped patches
        #self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

        #self.patch_hw = (img_size[1] // patch_size[1], img_size[0] // patch_size[0])
        #self.num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        _, _, h, w = self.get_output_shape(img_size) # n, emb_dim, h, w
        self.patch_hw = (h, w)
        self.num_patches = h*w

    def get_output_shape(self, img_size):
        # todo: don't be lazy..
        return self.proj(torch.randn(1,1,img_size[0],img_size[1])).shape 

    def forward(self, x):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        #assert H == self.img_size[0] and W == self.img_size[1], \
        #    f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

In [12]:
def inference(model, input_data, device):
    model.eval()
    input_data = input_data.to(device)

    with torch.no_grad():
        output = model(input_data)

    # return output.cpu().numpy()
    probabilities = torch.sigmoid(output)
    return probabilities.cpu().numpy()


In [ ]:
def main_inference():
    txt_path = testing_list_path
    with open(txt_path, 'r') as f:
        audio_list = [line.strip() for line in f]
    print("List of audio files to process:", audio_list)
    audio_conf = {
        'num_mel_bins': 128,
        'target_length': 128,
        'freqm': 0,
        'timem': 0,
        'mixup': 0,
        'dataset': 'birdsong',
        'mode': 'inference',
        'mean': -7.535187,
        'std': 2.8788178,
        'noise': False,
        'multilabel': True,
        'frame_shift_ms': 10,  # Added this line
        'low_freq': 100,
        'high_freq': 11000
    }

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = models_vit.__dict__['vit_base_patch16'](
    # model = models_vit.__dict__['vit_large_patch16'](
        num_classes=32,
        drop_path_rate=0.1,
        global_pool=True,
        mask_2d=True,
        use_custom_patch=False
    )

    img_size=(128, 128)
    in_chans= 1
    emb_dim = 768

    model.patch_embed = PatchEmbed_new(img_size=img_size, patch_size=(16,16), in_chans=in_chans, embed_dim=emb_dim, stride=16)
    num_patches = model.patch_embed.num_patches
    model.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, emb_dim), requires_grad=False)

    model_path = model_pth_path

    checkpoint = torch.load(model_path, map_location='cpu')
    model.load_state_dict(checkpoint['model'])
    model.to(device)

    audio_dir = Audio_path

    for audio_path in tqdm(glob.glob(os.path.join(audio_dir, '*.wav'))):
        base_name = os.path.basename(audio_path)
        file_name_without_extension = os.path.splitext(base_name)[0]
        if file_name_without_extension not in audio_list:
            continue

        dataset = load_input_data(audio_path, audio_conf)
        input_data = dataset[1][0].unsqueeze(0)
        probabilities = inference(model, input_data, device)

        all_probabilities = None
        all_segments = None
        for i in range(len(dataset)):
            input_data, _ = dataset[i]
            input_data = input_data.unsqueeze(0).to(device)
            probabilities = inference(model, input_data, device)
            probabilities_np = probabilities.squeeze()
            probabilities_np = probabilities_np.reshape(1, -1)
            if all_probabilities is None:
                all_probabilities = probabilities_np
                all_segments = np.array([[dataset[i][1], dataset.segments[i]['start_time'], dataset.segments[i]['end_time']]]) # Store the metadata with each segment
            else:
                all_probabilities = np.concatenate((all_probabilities, probabilities_np), axis=0)
                all_segments = np.concatenate((all_segments, np.array([[dataset[i][1], dataset.segments[i]['start_time'], dataset.segments[i]['end_time']]])), axis=0)

        df = pd.DataFrame(all_probabilities)
        df.columns = ['Label: ' + str(i) for i in range(all_probabilities.shape[1])]
        df['audio_path'] = all_segments[:, 0]
        df['start_time'] = all_segments[:, 1]
        df['end_time'] = all_segments[:, 2]
        df.index = range(1, len(df) + 1)
        df['sum'] = df.sum(axis=1)
        base_name = os.path.basename(audio_path)
        file_name_without_extension = os.path.splitext(base_name)[0]
        df = df[['audio_path', 'start_time', 'end_time'] + ['Label: ' + str(i) for i in range(all_probabilities.shape[1])] + ['sum']] # Arrange the columns as per your requirement
        relative_path = Path.cwd().parents[0].joinpath('report', 'model_new', 'inference', 'Raw_Output', f"{file_name_without_extension}.csv")
        df.to_csv(relative_path, sep=",", index_label="Index")

if __name__ == "__main__":
    main_inference()